# ⚡ ParagAI: High-Speed Local GPU Training & Deployment Guide

This notebook guides you through training and deploying **ParagAI** locally on your personal computer or workstation using an **NVIDIA GPU** (RTX 3060, 3070, 3080, 4070, 4090, or professional Tesla / Quadro cards).

> **💡 How to run under `.venv` in VS Code / IDE:**
> 1. Look at the **top right corner** of this notebook editor (where it displays the Python kernel name).
> 2. Click on it and choose **Select Another Kernel...** > **Python Environments...**.
> 3. Select **`.venv`** (`c:\Users\hedpa\Documents\AI\ParagAI\.venv\Scripts\python.exe`).
> *(Note: Even if you keep global Python selected, the code cells below automatically route all execution through `.venv`!)*

---

### 🌟 Capabilities Covered:
1. **Local NVIDIA CUDA & VRAM Inspection**
2. **Automated Diagnostic Suite (`doctor.py`)**
3. **Data Preparation & Replay Buffer Synthesis**
4. **Fast Indic + English BPE Tokenization**
5. **High-Throughput GPU Training (`bfloat16`, Gradient Accumulation, Live Telemetry)**
6. **Terminal & Python ChatML Inference**
7. **Automated Benchmark Evaluation (`eval.py`)**
8. **Export to Local Ollama & Local vLLM Server**

## 🔍 1. Verify Local NVIDIA GPU & Bind to `.venv`

In [ ]:
import os
import sys
import platform
import torch

# 1. Automatically change working directory to ParagAI repository root if started inside notebooks/
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

# 2. Add src to Python path
sys.path.insert(0, os.path.abspath("src"))

# 3. Locate and bind to the workspace virtual environment (.venv)
venv_py = os.path.abspath(os.path.join(".venv", "Scripts", "python.exe"))
if not os.path.exists(venv_py):
    venv_py = os.path.abspath(os.path.join(".venv", "bin", "python"))
py_exec = venv_py if os.path.exists(venv_py) else sys.executable

print("=" * 60)
print(f"Project Root:      {os.getcwd()}")
print(f"Using venv Python: {py_exec}")
print(f"PyTorch Version:   {torch.__version__}")
print(f"CUDA Available:    {torch.cuda.is_available()}")

if torch.cuda.is_available():
    dev_idx = torch.cuda.current_device()
    props = torch.cuda.get_device_properties(dev_idx)
    print(f"GPU Name:          {props.name}")
    print(f"Total VRAM:        {props.total_memory / (1024**3):.2f} GB")
    print(f"bfloat16 Ready:    {torch.cuda.is_bf16_supported()}")
else:
    print("WARNING: No CUDA GPU detected. Please check NVIDIA driver & PyTorch CUDA install.")
print("=" * 60)

## 🩺 2. Run ParagAI System Diagnostics (`doctor.py`)
Run the diagnostic tool under `.venv` to inspect your checkpoint files, tokenizer roundtrip fidelity, and Ollama service connectivity.

In [ ]:
!{py_exec} scripts/doctor.py

## 📚 3. Prepare Multi-Task SFT Data with Replay Mixing
Combines TinyStories, conversational chat, translations, and automation commands into a single high-quality corpus.

In [ ]:
# 3.1 Fetch sample TinyStories if not present
!{py_exec} scripts/fetch_tinystories.py --num_stories 2000 --output data/raw/stories/tinystories_local.txt

# 3.2 Prepare SFT stories with Replay Mix to avoid catastrophic forgetting
!{py_exec} scripts/prepare_tinystories_sft.py \
    --input data/raw/stories/tinystories_local.txt \
    --output data/processed/stories_sft.txt \
    --replay_mix True \
    --replay_ratio 0.15

# 3.3 Merge all multi-task files into combined corpus
!{py_exec} scripts/prepare_data.py --no_tags

## 🔤 4. Train Multilingual Indic + English BPE Tokenizer
Builds a robust BPE vocabulary with 100% 256-byte UTF-8 fallback coverage (Bengali, Hindi, English, Code, Emojis).

In [ ]:
!{py_exec} scripts/train_multilingual_bpe.py \
    --input "data/raw/**/*.txt" \
    --vocab_size 4096 \
    --output checkpoints/tokenizer_indic.json \
    --sample_mb 10

## 🚀 5. Local GPU Training with Live ASCII Telemetry
Trains on your local GPU with `configs/train_fast.yaml` (bfloat16 mixed precision and gradient accumulation).

In [ ]:
!{py_exec} scripts/train.py \
    --data data/processed/combined_corpus.txt \
    --config configs/train_fast.yaml \
    --model_config configs/model_small.yaml \
    --tokenizer_type bpe \
    --max_iters 1000

## 💬 6. Test Model Generation in Python

In [ ]:
import json
import os
import torch
from paragai.config import ModelConfig
from paragai.inference.generate import Generator
from paragai.model.transformer import ParagAIModel
from paragai.tokenizer.bpe_tokenizer import BPETokenizer
from paragai.tokenizer.char_tokenizer import CharTokenizer
from paragai.tokenizer.indic_bpe import IndicBPETokenizer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
checkpoint = torch.load('checkpoints/best_model.pt', map_location=device)
model_config = ModelConfig(**checkpoint['model_config'])
model = ParagAIModel(model_config)
state_dict = {k: v for k, v in checkpoint['model_state_dict'].items() if not k.endswith('.attn.mask')}
model.load_state_dict(state_dict, strict=False)

# Auto-detect tokenizer
tok_path = 'checkpoints/tokenizer.json' if os.path.exists('checkpoints/tokenizer.json') else 'checkpoints/tokenizer_indic.json'
with open(tok_path, 'r', encoding='utf-8') as f:
    tok_data = json.load(f)
if tok_data.get('tokenizer_class') == 'indic_bpe':
    tokenizer = IndicBPETokenizer.load(tok_path)
elif tok_data.get('type') == 'bpe':
    tokenizer = BPETokenizer.load(tok_path)
else:
    tokenizer = CharTokenizer.load(tok_path)

generator = Generator(model, tokenizer, device=device)

prompt = "<|im_start|>user\nHello! Tell me a story about a little bird.<|im_end|>\n<|im_start|>assistant\n"
response = generator.generate(prompt, max_new_tokens=100, temperature=0.3, include_prompt=False)
print(response.split('<|im_end|>')[0].strip())

## 📊 7. Run Benchmark Evaluation Suite (`eval.py`)

In [ ]:
!{py_exec} scripts/eval.py \
    --checkpoint checkpoints/best_model.pt \
    --tokenizer checkpoints/tokenizer.json \
    --tasks multiling,tools,story

## 📦 8. Export to Local Ollama & Deploy

In [ ]:
# 8.1 Export to SafeTensors and verify
!{py_exec} scripts/export_hf.py --checkpoint checkpoints/best_model.pt --output_dir checkpoints/export_hf --verify

# 8.2 Convert to GGUF with GPT-2 byte mapping
!{py_exec} scripts/export_gguf.py --input_dir checkpoints/export_hf --tokenizer checkpoints/tokenizer.json --output checkpoints/paragai.gguf

# 8.3 Register in Ollama (Make sure Ollama desktop app is running)
!ollama create paragai -f deploy/Modelfile

# 8.4 Test directly in Ollama
!ollama run paragai:latest "Hello! Who created you?"